# Cycle 4: APIs and Reproducible Acquisition

## 1. API Intro Classwork - pokeapi.co; Exploring the API Endpoints

### Endpoint 1: Contest Types

**Endpoint:** `GET https://pokeapi.co/api/v2/contest-type/{id or name}/`

**Route Parameters:** The route parameter is `{id or name}`. It allows the user to get a specific contest type using either its numerical ID (such as `1`, `2`, etc.) or its name (e.g. `cool`, `beautiful`, etc.)

**Data Retrieved:** This endpoint returns information about a Pokémon contest type, including its ID, name, associated berry flavor, and names in different langauges. One example is the "cool" contest type has a corresponding ID of `1`, and is associated with the "spicy" berry flavor, and is associated with the color "red". 

### Endpoint 2: Currencies

**Endpoint:** `GET https://pokeapi.co/api/v2/currency/{id or name}/`

**Route Parameters:** The route parameter is `{id or name}`. It allows the user to get a specific currency using either its numerical ID (e.g., `1`) or its name (e.g., `poke-dollar`).

**Data Retrieved:** This endpoint returns information about a currency used in Pokémon games, including its ID, name, and names in different languages. For example, the "poke-dollar" currency has an ID of `1` and is called "Poké Dollar" in English.


### Pokémon Contest Type Keys

Four main keys:

- **`id`**: The number identifying the contest type
- **`name`**: The name of the contest type (e.g., `tough`)
- **`berry_flavor`**: A dictionary containing two keys: name (the berry flavor, sour) and url (a link to the berry flavor's API endpoint)
- **`names`**: A list of dictionaries containing the contest type's names in different languages. Each dictionary has a name, color, and language. The language key contains another dictionary with the language's name and url

## 2. Load and Explore Saved JSON Fixtures

I loaded the two saved API fixtures using `json.load()`. These fixtures let me inspect the data without connecting directly to the API. I printed the dictionary keys and selected values to see what each endpoint contains.

In [1]:
from pathlib import Path
import json
import requests

# Load contest type data
contest_path = Path("contest_type.json")
with contest_path.open(encoding="utf-8") as handle:
    contest_data = json.load(handle)

print(contest_data.keys())

# Load currency data
currency_path = Path("currency.json")
with currency_path.open(encoding="utf-8") as handle:
    currency_data = json.load(handle)

print(currency_data.keys())


dict_keys(['id', 'name', 'berry_flavor', 'names'])
dict_keys(['id', 'name', 'names'])


In [2]:
# Contest type
print("Contest Type:", contest_data["name"])
print("ID:", contest_data["id"])
print("Berry Flavor:", contest_data["berry_flavor"]["name"])

# Currency
print("Currency:", currency_data["name"])
print("ID:", currency_data["id"])

Contest Type: tough
ID: 5
Berry Flavor: sour
Currency: poke-dollar
ID: 1


## 3. Define Endpoints and Expected Structure

I chose the `berry_flavor` dictionary from Contest Type and the `names` list from Currency. I defined the API endpoints and parameter dictionaries to record the ID used for each request. PokéAPI includes IDs directly in the URL paths, so the selected ID of 1 is already part of each endpoint. I expect `berry_flavor` to contain `name` and `url`, and I used the saved fixtures to determine the expected list lengths.

In [3]:
# API endpoints
CONTEST_ENDPOINT = "https://pokeapi.co/api/v2/contest-type/1/"
CURRENCY_ENDPOINT = "https://pokeapi.co/api/v2/currency/1/"

# Parameters for each endpoint
CONTEST_PARAMS = {"id": 1}
CURRENCY_PARAMS = {"id": 1}

# Expected keys in the berry_flavor dictionary
BERRY_FLAVOR_KEYS = {"name", "url"}

# Expected number of translated names from fixtures
EXPECTED_LENGTH = len(contest_data["names"])
CURRENCY_EXPECTED_LENGTH = len(currency_data["names"])


## 4. Validate the JSON Structure

I validated endpoints Contest Type and Currency.

**Dictionary Validation (Contest Type):** Checked that the overall response and `berry_flavor` are dictionaries. I then used set subtraction to compare the expected keys (`name` and `url`) with the actual keys in `berry_flavor`. If any keys are missing, the function raises a `ValueError`.

**List Validation (Currency):** I checked that `names` exists and is a list. I used `len()` to compare its length against the expected length from the saved currency fixture. If the lengths differ, the function raises a `ValueError`.


In [4]:
def validate_contest_data(payload):
    # Check that the overall response is a dictionary
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")

    # Check that berry_flavor exists and is a dictionary
    if "berry_flavor" not in payload or not isinstance(payload["berry_flavor"], dict):
        raise ValueError("Missing berry_flavor object")

    # Compare the expected keys with the actual keys in berry_flavor
    missing = BERRY_FLAVOR_KEYS - set(payload["berry_flavor"])

    # Raise an error if any expected keys are missing
    if missing:
        raise ValueError(f"Missing berry_flavor fields: {sorted(missing)}")

    # Check that names exists and is a list
    if "names" not in payload or not isinstance(payload["names"], list):
        raise ValueError("Missing names list")

    # Find the length of the names list
    length = len(payload["names"])

    # Check if the length matches the saved fixture
    if length != EXPECTED_LENGTH:
        raise ValueError(f"Names list has incorrect length: {length}")

    # Return the payload if all validation checks pass
    return payload

def validate_currency_data(payload):
    # Check that the overall response is a dictionary
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")

    # Check that names exists and is a list
    if "names" not in payload or not isinstance(payload["names"], list):
        raise ValueError("Missing currency names list")

    # Find the length of the currency names list
    length = len(payload["names"])

    # Check if the length matches the saved currency fixture
    if length != CURRENCY_EXPECTED_LENGTH:
        raise ValueError(f"Currency names list has incorrect length: {length}")

    # Return the payload if validation passes
    return payload

## 5. Retrieve Live Data and Handle Errors

**Timeout and HTTP Status:** I used `timeout=10` to prevent the program from waiting indefinitely for a response. I used `raise_for_status()` to detect HTTP errors, including 4xx client errors (such as 404 Not Found) and 5xx server errors (such as 500 Internal Server Error). A 200 status indicates a successful request.

source: https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status

**Error Handling:** I used `try` and `except` to catch connection, timeout, HTTP, JSON, and validation errors. If the live request fails or returns data that doesn't match the expected structure, the program uses the saved fixture instead.

In [5]:
# Contest Type API
def acquire_contest(use_live=False):
    source = "cached fixture"
    payload = contest_data
    if use_live:
        try:
            # Send a GET request to the API. Stop waiting if the API connection or response takes too long
            response = requests.get(CONTEST_ENDPOINT, timeout=10)
            # Check the HTTP status code. Raise an error for HTTP error statuses (4xx or 5xx)
            # 200 means the request succeeded.
            # 4xx codes indicate client errors, such as 404 Not Found.
            # 5xx codes indicate server errors, such as 500 Internal Server Error.
            # raise_for_status() raises an exception for 4xx and 5xx errors.
            response.raise_for_status()
            payload = response.json()
            # Check that the response has the expected structure

            validate_contest_data(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = contest_data
    return validate_contest_data(payload), source

# Currency API
def acquire_currency(use_live=False):
    source = "cached fixture"
    payload = currency_data
    if use_live:
        try:
            # Send a GET request; timeout=10 prevents waiting indefinitely for a response
            response = requests.get(CURRENCY_ENDPOINT, timeout=10)
            # Raise an error for HTTP error statuses (4xx or 5xx)
            response.raise_for_status()
            # Convert the JSON response to Python data
            payload = response.json()

            validate_currency_data(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = currency_data
    return validate_currency_data(payload), source

## 6. Test the Fixture and Live API Responses

I first tested the saved fixtures, then attempted live requests. The acquisition functions return the validated payload and its source. If the live request fails, they use the saved fixture instead. I print the sources and selected values to check the results.

In [6]:
# Test both saved fixtures 
contest_fixture, contest_fixture_source = acquire_contest(use_live=False)
currency_fixture, currency_fixture_source = acquire_currency(use_live=False)

# Live API data; use fixtures if a request or validation fails
contest_live, contest_source = acquire_contest(use_live=True)
currency_live, currency_source = acquire_currency(use_live=True)

# Print the data source
print("Contest source:", contest_source)
print("Currency source:", currency_source)

# Validated responses
print("Contest Type:")
print(contest_live)
print("Currency:")
print(currency_live)

# Access specific values
print("Contest Type:", contest_live["name"])
print("Berry Flavor:", contest_live["berry_flavor"]["name"])
print("Contest Names:", len(contest_live["names"]))
print("Currency:", currency_live["name"])
print("Currency Names:", len(currency_live["names"]))

Contest source: https://pokeapi.co/api/v2/contest-type/1/
Currency source: https://pokeapi.co/api/v2/currency/1/
Contest Type:
{'id': 1, 'name': 'cool', 'berry_flavor': {'name': 'spicy', 'url': 'https://pokeapi.co/api/v2/berry-flavor/1/'}, 'names': [{'name': 'かっこよさ', 'color': '', 'language': {'name': 'ja-hrkt', 'url': 'https://pokeapi.co/api/v2/language/1/'}}, {'name': '帥氣', 'color': '紅', 'language': {'name': 'zh-hant', 'url': 'https://pokeapi.co/api/v2/language/4/'}}, {'name': 'Sang-froid', 'color': 'Rouge', 'language': {'name': 'fr', 'url': 'https://pokeapi.co/api/v2/language/5/'}}, {'name': 'Cool', 'color': 'Red', 'language': {'name': 'en', 'url': 'https://pokeapi.co/api/v2/language/9/'}}, {'name': '帅气', 'color': '红', 'language': {'name': 'zh-hans', 'url': 'https://pokeapi.co/api/v2/language/12/'}}]}
Currency:
{'id': 1, 'name': 'poke-dollar', 'names': [{'name': 'Pokémon Dollar', 'language': {'name': 'en', 'url': 'https://pokeapi.co/api/v2/language/9/'}}]}
Contest Type: cool
Berry Fl

## 7. Normalize the JSON into a DataFrame

Both endpoints have a `names` list containing translated names. I used `pd.DataFrame()` to convert each list into a separate DataFrame. Since the `language` column originally contained nested dictionaries, I extracted the language names to make the tables easier to read and analyze.

In [7]:
import pandas as pd

# Normalize data
contest_df = pd.DataFrame(contest_live["names"])
currency_df = pd.DataFrame(currency_live["names"])

# Extract language names from nested dictionaries
contest_df["language"] = [item["name"] for item in contest_df["language"]]

currency_df["language"] = [item["name"] for item in currency_df["language"]]

# Check the data
assert len(contest_df) > 0
assert len(currency_df) > 0

contest_df


,name,color,language
0,かっこよさ,,ja-hrkt
1,帥氣,紅,zh-hant
2,Sang-froid,Rouge,fr
3,Cool,Red,en
4,帅气,红,zh-hans


## 8. Check the DataFrame

I check that both DataFrames have the expected number of rows based on the translated names in each API response. I also check that the `name` and `language` columns exist.

In [8]:
# Check the number of rows
assert len(contest_df) == len(contest_live["names"])
assert len(currency_df) == len(currency_live["names"])

# Check the expected columns
assert "name" in contest_df.columns
assert "language" in contest_df.columns

assert "name" in currency_df.columns
assert "language" in currency_df.columns

print("Contest Type rows:", len(contest_df))
print("Currency rows:", len(currency_df))

print("Contest Type columns:", contest_df.columns)
print("Currency columns:", currency_df.columns)

# Check for missing values
print("Contest missing values:")
print(contest_df.isna().sum())

print("Currency missing values:")
print(currency_df.isna().sum())

# Check for duplicate rows
print("Contest duplicates:", contest_df.duplicated().sum())
print("Currency duplicates:", currency_df.duplicated().sum())

# Check IDs are positive
assert contest_live["id"] > 0
assert currency_live["id"] > 0


contest_df

Contest Type rows: 5
Currency rows: 1
Contest Type columns: Index(['name', 'color', 'language'], dtype='str')
Currency columns: Index(['name', 'language'], dtype='str')
Contest missing values:
name        0
color       0
language    0
dtype: int64
Currency missing values:
name        0
language    0
dtype: int64
Contest duplicates: 0
Currency duplicates: 0


,name,color,language
0,かっこよさ,,ja-hrkt
1,帥氣,紅,zh-hant
2,Sang-froid,Rouge,fr
3,Cool,Red,en
4,帅气,红,zh-hans


## 9. Record Data Provenance

I recorded the API provider, endpoints, parameters, and acquisition sources for both datasets. I also included the saved fixture filenames, retrieval date, and attribution information to document where the data came from.

In [9]:

provenance = {
    "provider": "PokéAPI",
    "contest_endpoint": CONTEST_ENDPOINT,
    "currency_endpoint": CURRENCY_ENDPOINT,
    "contest_parameters": CONTEST_PARAMS,
    "currency_parameters": CURRENCY_PARAMS,
    "contest_acquisition_source": contest_source,
    "currency_acquisition_source": currency_source,
    "contest_fixture": "contest_type.json",
    "currency_fixture": "currency.json",
    "fixture_retrieved": "11/12/2026",
    "license": "See PokéAPI documentation",
}

provenance


{'provider': 'PokéAPI',
 'contest_endpoint': 'https://pokeapi.co/api/v2/contest-type/1/',
 'currency_endpoint': 'https://pokeapi.co/api/v2/currency/1/',
 'contest_parameters': {'id': 1},
 'currency_parameters': {'id': 1},
 'contest_acquisition_source': 'https://pokeapi.co/api/v2/contest-type/1/',
 'currency_acquisition_source': 'https://pokeapi.co/api/v2/currency/1/',
 'contest_fixture': 'contest_type.json',
 'currency_fixture': 'currency.json',
 'fixture_retrieved': '11/12/2026',
 'license': 'See PokéAPI documentation'}

## 10. Observation and Reflection

**Observation:** Both endpoints have a `names` list containing translations in different languages. I converted each list into a separate DataFrame using `pd.DataFrame()`. The Contest Type DataFrame has 5 rows and 3 columns (`name`, `color`, and `language`), while the Currency DataFrame has 1 row and 2 columns (`name` and `language`). I extracted the language names from the nested dictionaries to make the data easier to read and analyze.

### What changes between live and archived data?

Live data comes directly from the API and may change over time, while archived data is a saved version that stays the same. In my notebook, I used live PokéAPI data for Contest Types and Currencies and compared it with saved JSON fixtures. Live data depends on the API and internet connection, while the fixtures allow me to access the same data even when the API is unavailable.

### Which check prevents the most misleading failure?

I think the dictionary-key validation is the most important because it checks whether the API response contains the information my code needs. For example, my `validate_contest_data()` function checks that `berry_flavor` contains both `name` and `url`. If either is missing, the function raises a `ValueError` instead of continuing with incomplete data. I also check the lengths of the `names` lists for both endpoints to make sure they match the saved fixtures.

### How does caching support respectful use and reproducibility?

Caching lets me reuse saved API data instead of making unnecessary requests to the server. This reduces traffic to PokéAPI and makes my notebook more reliable. In my notebook, setting `use_live=False` loads the fixtures directly, and if a live request fails, the functions automatically fall back to the saved data.